# Phase 2 — Data Cleaning

**Input**: `reference_database/extracted_raw/*.csv` (21 files, 2467 rows)
**Output**: `reference_database/cleaned/*.csv` and master `all_sources_cleaned.csv`

## Fixes applied
1. Khalidi 2009: `Unnamed: 2` -> BingolA (replicate); `Meydan_Dag˘` -> MeydanDag
2. Yellin & Perlman 1981: Re-extracted with Row 1 headers (GLD/HTMD/KRUD ...); mean parsed from ±sd
3. Yellin & Perlman 1980: Beisamoun + Nahal Lavan flagged `is_source_reference=False`
4. Milic 2014: 7 "Mean" meta-rows removed
5. Frahm 2013: All 65 rows flagged `is_source_reference=False` (Tell Mozan artefacts)
6. Frahm & Hauck 2017 main: Average rows dropped; artifact rows flagged False
7. Rosenberg & Carter 2022: Sub-outcrop labels normalised
8. Carter 2013 Kenan Tepe: Bingol split BingolA (Sr<10 ppm) / BingolB (Sr>=10 ppm)
9. Oxide wt% columns noted — NOT converted to elemental ppm
10. Master `all_sources_cleaned.csv` built: 2375 rows

## Method tier key
| Tier | Method | pXRF-comparable |
|------|--------|-----------------|
| 1 | LA-ICP-MS / ICP-MS / ICP-AES | Yes (trace elements) |
| 2 | EDXRF / WDXRF (lab) | Yes |
| 3 | pXRF (portable) | Direct |
| 4 | NAA / INAA | No (REE elements only) |


In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import re
from pathlib import Path

ROOT    = Path(r"c:\work\code\obsidian")
RAW_DIR = ROOT / "reference_database" / "extracted_raw"
CLN_DIR = ROOT / "reference_database" / "cleaned"
CLN_DIR.mkdir(exist_ok=True)

# Abbreviated source map (see full map in 01_data_extraction.ipynb)
SOURCE_MAP = {
    "east gollu dag": "EGD", "gollu dag east": "EGD", "egd": "EGD", "gld": "EGD",
    "west gollu dag": "WGD", "gollu dag west": "WGD", "wgd": "WGD",
    "gollu dag": "GolluDag",
    "nenezi dag": "ND", "nenezi": "ND", "nd": "ND", "nnzd": "ND",
    "bingol a": "BingolA", "bingol-a": "BingolA",
    "bingol b": "BingolB", "bingol-b": "BingolB",
    "bingol": "Bingol",
    "nemrut dag": "NemrutDag", "nmrd1": "NemrutDag", "nmrd2": "NemrutDag", "nemrut dag a": "NemrutDag",
    "meydan dag": "MeydanDag",
    "pasinler": "Pasinler", "pasinler tizgi": "Pasinler", "pasinler eksisu": "Pasinler",
    "sarikamis": "Sarikamis", "sarakamis hamamli": "Sarikamis", "sarakamis sehetin": "Sarikamis",
    "suphan dag": "SuphanDag", "gurgurbabatepe": "SuphanDag",
    "sevan": "Sevan", "znkt": "Sevan",
    "htmd": "HasanDag", "hasan dag": "HasanDag",
    "krud": "KRUD", "acigol": "AciGol",
}

def nrm(raw):
    """Normalise source label: NFC unicode, lowercase, strip, SOURCE_MAP lookup."""
    if pd.isna(raw): return np.nan
    key = unicodedata.normalize("NFC", str(raw)).strip().lower()
    return SOURCE_MAP.get(key, unicodedata.normalize("NFC", str(raw)).strip())

def parse_pm(val):
    """Parse mean from 174.3+/-5.6 notation. Returns NaN if unparsable."""
    if pd.isna(val): return np.nan
    s = str(val).strip()
    m = re.match(r"^([\d.]+)\s*[\xb1\u00b1]", s)
    if m: return float(m.group(1))
    try: return float(s)
    except: return np.nan

print("Phase 2 ready.", len(SOURCE_MAP), "source map entries")


---
## Fix 1 — Khalidi & Gratuze 2009

In [ ]:
df_k = pd.read_csv(RAW_DIR / "khalidi_gratuze_2009.csv")
print("Before:", df_k[["source_raw","source"]].to_string())

# "Unnamed: 2" = duplicate BingolA column in original xlsx
df_k.loc[df_k["source_raw"] == "Unnamed: 2", "source"] = "BingolA"
df_k.loc[df_k["source_raw"] == "Unnamed: 2", "source_raw"] = "Bingol_A (replicate)"

# Meydan_Dag˘ has U+02D8 BREVE — not in unicode map; fix directly
meydan_mask = df_k["source_raw"].str.contains("Meydan", case=False, na=False)
df_k.loc[meydan_mask, "source"] = "MeydanDag"

df_k["is_source_reference"] = True
df_k.to_csv(CLN_DIR / "khalidi_gratuze_2009.csv", index=False)
print("After:", df_k["source"].tolist())


---
## Fix 2 — Yellin & Perlman 1981 (re-extract from xlsx)

**Bug**: Phase 1 used `iloc[0]` which gave region labels (Central, Anatolia, Eastern, Turkey).
The actual source abbreviations are in Row 1: GLD, HTMD, KRUD, NNZD, NMRD1, NMRD2, ZNKT, Sevan.

**Source code mapping**:
- GLD → EGD (Göllü Dağ East)
- HTMD → HasanDag
- KRUD → KRUD (sub-source, identity uncertain)
- NNZD → ND (Nenezi Dağ)
- NMRD1, NMRD2 → NemrutDag
- ZNKT, Sevan → Sevan

Note: NNZD values are in `mean±sd` format — parse the mean only.

In [ ]:
d2 = ROOT / "obsidian_minerales_component_tables_from_articles" / "data2.xlsx"
raw = pd.read_excel(d2, sheet_name="Yellin and Perlman 1981", header=None, dtype=str)
raw = raw.dropna(how="all").dropna(axis=1, how="all")

# Row 0 = region group labels (skip); Row 1 = source abbreviations (use)
abbrevs = raw.iloc[1, 1:].tolist()
# Only iterate rows where the element name (col 0) is not NaN
el_rows = [(i+2, str(raw.iloc[i+2, 0]).strip())
           for i in range(len(raw)-2) if not pd.isna(raw.iloc[i+2, 0])]
print("Sources:", abbrevs)
print("Elements:", [e for _,e in el_rows])

rows = []
for si, abbr in enumerate(abbrevs, start=1):
    rec = {"source_raw": str(abbr).strip(), "source": nrm(str(abbr).strip())}
    for ri, el in el_rows:
        if ri < len(raw):
            rec[el] = parse_pm(raw.iloc[ri, si])
    rows.append(rec)

df_y81 = pd.DataFrame(rows)
df_y81["method"] = "NAA/INAA"
df_y81["method_tier"] = 4
df_y81["year"] = 1981
df_y81["paper"] = "Yellin & Perlman 1981"
df_y81["units"] = "ppm"
df_y81["is_source_reference"] = True
df_y81.to_csv(CLN_DIR / "yellin_perlman_1981.csv", index=False)
print(df_y81[["source_raw", "source", "La", "Ce"]])


---
## Fix 3 — Yellin & Perlman 1980 (flag artefact rows)

`Beisamoun` and `Nahal Lavan` = Israeli Neolithic sites. Their obsidian chemistry matches Gollu Dag.
They are archaeological artefact measurements, NOT geological source reference samples.
Flag `is_source_reference=False` and set `attributed_source=GolluDag`.

In [ ]:
df_y80 = pd.read_csv(RAW_DIR / "yellin_perlman_1980.csv")
GEO_SOURCES = {"GolluDag", "ND"}
df_y80["is_source_reference"] = df_y80["source"].isin(GEO_SOURCES)

# Use loop to set string values (numpy cannot broadcast str into float column)
df_y80["site"] = df_y80["site"].astype(object)
df_y80["attributed_source"] = ""
for idx in df_y80[df_y80["source"].isin({"Beisamoun","Nahal Lavan"})].index:
    df_y80.at[idx, "site"] = df_y80.at[idx, "source"]
    df_y80.at[idx, "attributed_source"] = "GolluDag"

print(df_y80[["source","is_source_reference","site"]].to_string())
df_y80.to_csv(CLN_DIR / "yellin_perlman_1980.csv", index=False)


---
## Fixes 4-8 and pass-through

In [ ]:
# -- Fix 4: Milic 2014 — drop "Mean" meta-rows --
df_ml = pd.read_csv(RAW_DIR / "milic_2014.csv")
df_ml = df_ml[df_ml["source"] != "Mean"].copy().reset_index(drop=True)
df_ml["is_source_reference"] = True
df_ml.to_csv(CLN_DIR / "milic_2014.csv", index=False)
print("Milic 2014:", df_ml["source"].value_counts().to_dict())

# -- Fix 5: Frahm 2013 — flag as artefact data --
df_fr13 = pd.read_csv(RAW_DIR / "frahm_2013.csv")
df_fr13["is_source_reference"] = False
df_fr13["notes"] = "Archaeological artefacts (Tell Mozan debitage) — pXRF validity study"
df_fr13.to_csv(CLN_DIR / "frahm_2013.csv", index=False)
print("Frahm 2013:", len(df_fr13), "rows flagged is_source_reference=False")

# -- Fix 6: Frahm & Hauck 2017 main — drop Average, flag artifact rows --
df_fh = pd.read_csv(RAW_DIR / "frahm_hauck_2017_main.csv")
df_fh = df_fh[df_fh["source"] != "Average"].copy()
df_fh["is_source_reference"] = ~(df_fh["source"] == "artifact")
df_fh.to_csv(CLN_DIR / "frahm_hauck_2017_main.csv", index=False)
print("Frahm&Hauck 2017:", df_fh["is_source_reference"].value_counts().to_dict())

# -- Fix 7: Rosenberg 2022 — normalise sub-outcrop labels --
# Note: this CSV has SOURCE column only (no source_raw) — re-map from source
df_rc = pd.read_csv(RAW_DIR / "rosenberg_carter_2022_sources.csv")
df_rc["source"] = df_rc["source"].apply(nrm)
df_rc["is_source_reference"] = True
df_rc.to_csv(CLN_DIR / "rosenberg_carter_2022_sources.csv", index=False)
print("Rosenberg 2022:", df_rc["source"].value_counts().to_dict())

# -- Fix 8: Carter 2013 Kenan Tepe — split Bingol A/B by Sr discriminant --
# Reference: BingolA Sr ~ 0-5 ppm (Khalidi 2009); BingolB Sr ~ 30-55 ppm
# Threshold of 10 ppm is clear from bimodal distribution in this dataset
df_ca = pd.read_csv(RAW_DIR / "carter_2013_kenan_tepe.csv")
bm = df_ca["source"] == "Bingol"
sr = pd.to_numeric(df_ca["Sr"], errors="coerce")
df_ca.loc[bm & (sr < 10), "source"] = "BingolA"
df_ca.loc[bm & (sr >= 10), "source"] = "BingolB"
df_ca["is_source_reference"] = True
df_ca.to_csv(CLN_DIR / "carter_2013_kenan_tepe.csv", index=False)
print("Carter 2013:", df_ca["source"].value_counts().to_dict())

# -- Pass-through: copy remaining raw CSVs with is_source_reference flag --
DONE = {"khalidi_gratuze_2009.csv","yellin_perlman_1981.csv","yellin_perlman_1980.csv",
        "milic_2014.csv","frahm_2013.csv","frahm_hauck_2017_main.csv",
        "rosenberg_carter_2022_sources.csv","carter_2013_kenan_tepe.csv"}
for fp in sorted(RAW_DIR.glob("*.csv")):
    if fp.name in DONE: continue
    df = pd.read_csv(fp)
    if "is_source_reference" not in df.columns: df["is_source_reference"] = True
    df.to_csv(CLN_DIR / fp.name, index=False)
    print("Pass-through:", fp.name, len(df), "rows")


---
## Build master `all_sources_cleaned.csv`

Merge all cleaned source-reference rows. Keep only the 14 pXRF-compatible elements.

In [ ]:
PXRF_EL = ["Rb","Sr","Zr","Nb","Y","Fe","Mn","Ba","Zn","Ti","Th","U","Pb","Ga"]
META = ["sample_id","source","source_raw","site","paper","year",
        "method","method_tier","units","is_source_reference","notes","verification_flag"]

frames = []
for fp in sorted(CLN_DIR.glob("*.csv")):
    if fp.name == "all_sources_cleaned.csv": continue  # skip master itself
    df = pd.read_csv(fp)
    if "is_source_reference" in df.columns:
        df = df[df["is_source_reference"] == True].copy()
    if len(df) == 0: continue
    keep = [c for c in META if c in df.columns]
    els  = [c for c in PXRF_EL if c in df.columns]
    sub = df[keep+els].copy()
    for el in PXRF_EL:
        if el not in sub.columns: sub[el] = np.nan
    sub["source_file"] = fp.name
    frames.append(sub)

df_all = pd.concat(frames, ignore_index=True)
df_all["source"] = df_all["source"].apply(nrm)
df_all.to_csv(CLN_DIR / "all_sources_cleaned.csv", index=False)

print(f"Master: {len(df_all)} rows")
print("\nSource distribution:")
print(df_all["source"].value_counts().to_string())
